# Badminton Analysis Pipeline (Colab)
Runs the stabilized multi-trajectory pipeline on your hand-held phone match videos.

**Before running:** `Runtime ▸ Change runtime type ▸ GPU` (T4). All inputs are pulled from Google Drive (no manual uploads).

## 1. Clone + install

In [ ]:
!git clone https://github.com/sujkuttan/baddyAnalysis.git
%cd baddyAnalysis
!pip install -r requirements.txt
!pip install -q ipyevents

## 2. Download match video from Google Drive
Edit `VIDEO_ID` if the link changes. The file id is the part after `file/d/` in the share URL.

In [ ]:
!pip install -q gdown
VIDEO_ID = "1aA_3keNIfCBjkNC9isovHwTQjfVejQdC"
!gdown {VIDEO_ID} -O match.mp4
video_name = "match.mp4"
print("video:", video_name)

## 3. Click 4 court corners (interactive)
The frame is shown below. **Click exactly 4 court corners in order: TL, TR, BR, BL.**
Clicks are captured automatically (coordinates scaled to the original image) and saved to `corners.json`.

In [ ]:
import cv2, json, ipywidgets as widgets, ipyevents
from PIL import Image
from io import BytesIO

img = cv2.cvtColor(cv2.imread('data/first_frame.jpg'), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
disp_w = 800
scale = disp_w / w
disp_h = int(h * scale)

pil = Image.fromarray(img).resize((disp_w, disp_h))
buf = BytesIO(); pil.save(buf, 'png')
img_w = widgets.Image(value=buf.getvalue(), format='png', width=disp_w, height=disp_h)
ev = ipyevents.Events(img_w, watched_events=['click'])
points = []

def handle(event):
    if len(points) >= 4:
        return
    ex = event['data']['offsetX']; ey = event['data']['offsetY']
    nx = int(ex / disp_w * w); ny = int(ey / disp_h * h)
    points.append((nx, ny))
    print('corner', len(points), ':', (nx, ny))
    if len(points) == 4:
        json.dump({'corners': [list(p) for p in points]}, open('corners.json', 'w'))
        print('saved corners.json:', points)

ev.on_dom_event(handle)
display(img_w)
print('Click TL, TR, BR, BL on the image above')

## 4. Download TrackNet weights (zip) from Google Drive
The zip includes TrackNet + inpaint weights. It is unzipped into `weights/` and the
`.pt` is located automatically.

In [ ]:
import os, glob
!gdown 1rhKXbff1GITgrFTYptW6gAvWZ76E_qzp -O tracknet.zip
!unzip -o tracknet.zip -d weights/
cands = glob.glob('weights/**/TrackNet*.pt', recursive=True)
tracknet = cands[0] if cands else None
print('tracknet weights:', tracknet)
print('weights dir:', os.listdir('weights'))

## 5. Run the pipeline
Produces `data/new_predictions.csv`, `data/metrics.json`, `data/annotated.mp4`,
`data/coverage_heatmap.png`, `data/fatigue.png`, `data/coaching_report.md`.
Fine-tunes on `labels_import.csv` if present.

In [ ]:
from src import pipeline
corners = json.load(open('corners.json'))['corners']
res = pipeline.run_full_pipeline(
    video_name, corners, out_dir='data',
    labels_csv='labels_import.csv', device='cuda',
    tracknet_weights=tracknet,
)
print('predictions:', res['predictions_csv'])
print('metrics:', res['metrics'])

## 6. A/B vs your BST pipeline
Export your BST predictions to `bst_predictions.csv` (columns `frame,predicted_stroke`),
upload it via `files.upload()`, then compare. The `labeled` rows in `labels_import.csv`
are the shared ground truth. If you don't have BST preds yet, skip this cell.

In [ ]:
from google.colab import files
# files.upload()  # upload bst_predictions.csv  (optional)
from src import pipeline
bst = 'bst_predictions.csv' if os.path.exists('bst_predictions.csv') else None
pipeline.ab_compare('labels_import.csv', 'data/new_predictions.csv', bst)